# When is the GPU worth it — a performance model for the likelihood backend

The GPU backend evaluates the likelihood on the device and leaves the sampler, the search
and the evidence bookkeeping untouched. So the question is not *whether* the GPU is faster
at the likelihood — it is, 8× in double precision on an A100 against 16 EPYC cores — but
**when that reaches the wall clock**. Three things decide it:

1. **Data size** `n`: a GPU launch has a fixed cost (~30 µs) that a 1000-point likelihood does
   not amortise, a 100 000-point one does. Measured on a RTX 4090: ~20 µs per launch, and the
   host threads serialise on the driver, so launches do not overlap.
2. **Batch size** `B`: how many parameter vectors one launch evaluates. One per launch (today)
   pays the launch cost every time; batching the search's trials pays it once per ~100.
3. **The likelihood's share of the run** `f`: Amdahl. At K=10 000 it is 53 % of CPU time,
   at K=64 000 it is 91 %. Nothing done to the likelihood can beat `1/(1-f)`.

Every number below is anchored on a measurement from 2026-10-23 (`test_12`, two gaussians,
1000 points, fp64), and every assumption is a named parameter you can move.

In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # embeds plotly.js: renders in any JupyterLab, no extension needed

BLUE, ORANGE, GREY, INK, SURFACE, GRID = "#2a78d6", "#eb6834", "#6b6b68", "#1a1a19", "#fcfcfb", "#e6e6e3"

# --- measured anchors -----------------------------------------------------------------
CPU_RATE_16C = 0.416e6                        # likelihood evaluations / s, EPYC-Genoa 16 threads
GPU_RATE_SAT = {"A100 (fp64 1:2)": 3.37e6,    # saturating batch (8192 threads in flight)
                "L40S (fp64 1:64)": 1.46e6, "V100 (fp64 1:2)": 1.11e6}
TRIALS_PER_ACCEPT = 108                       # 161 M calls / 1.49 M accepted points at K=64000
LIKELIHOOD_FRACTION = {"K=10000": 0.53, "K=64000": 0.91}

# --- the model --------------------------------------------------------------------------
def cpu_eval_us(n, c0=2.0, c1=0.0365):            # one core: 2 us + 36.5 ns per point
    return c0 + c1 * n
def cpu_rate(n, threads=16):
    return threads / (cpu_eval_us(n) * 1e-6)
def gpu_launch_us(n, batch, card="A100 (fp64 1:2)", launch_us=20.0, block_us=2.0, concurrent_blocks=864):
    g  = 1e6 / (GPU_RATE_SAT[card] * 1000.0)                          # us / point / evaluation, saturated
    b1 = 2.5e-4 * GPU_RATE_SAT["A100 (fp64 1:2)"] / GPU_RATE_SAT[card] # us / point for one block
    waves = np.ceil(np.asarray(batch, float) / concurrent_blocks)
    return launch_us + np.maximum((block_us + n * b1) * waves, batch * n * g)
def gpu_rate(n, batch, in_flight=1, **kw):
    return in_flight * batch / (gpu_launch_us(n, batch, **kw) * 1e-6)
def likelihood_speedup(n, batch, threads=16, in_flight=1, **kw):
    return gpu_rate(n, batch, in_flight, **kw) / cpu_rate(n, threads)
def end_to_end(speedup, fraction):
    return 1.0 / ((1.0 - fraction) + fraction / speedup)

def style(fig, title, xlab, ylab, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h,
                      plot_bgcolor=SURFACE, paper_bgcolor=SURFACE, font=dict(color=INK),
                      margin=dict(t=50, b=40, l=10, r=10), hovermode="x unified")
    fig.update_xaxes(gridcolor=GRID); fig.update_yaxes(gridcolor=GRID)
    return fig

## 1. Throughput against data size

Likelihood evaluations per second. The CPU line is 16 cores. The GPU lines differ only in
how many evaluations share one launch. Below the crossing the CPU wins.

In [7]:
n = np.logspace(2, 6, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=n, y=cpu_rate(n), name="CPU, 16 cores", line=dict(color=GREY, width=2)))
series = [("GPU, 1 per launch (today)",              dict(batch=1, in_flight=1),  BLUE, "dot"),
          ("GPU, batch 16 (threads' candidates)",     dict(batch=16),             BLUE, "dash"),
          ("GPU, batch 108 (slice trials)",           dict(batch=TRIALS_PER_ACCEPT), BLUE, "solid"),
          ("GPU, batch 1024",                         dict(batch=1024),           ORANGE, "solid")]
for name, kw, col, dash in series:
    fig.add_trace(go.Scatter(x=n, y=gpu_rate(n, **kw), name=name, line=dict(color=col, width=2, dash=dash)))
fig.update_xaxes(type="log"); fig.update_yaxes(type="log")
style(fig, "Likelihood evaluations per second, double precision, A100 vs 16 EPYC cores",
      "data points per likelihood (n)", "evaluations / s")

Read it as: **today's design crosses the CPU around a few thousand points**; with the slice
trials batched the crossing is near 100 points and the GPU line approaches its saturated 8×.
Batching changes *where* the GPU wins; data size changes *by how much*.

## 2. Break-even data size against batch size

The smallest `n` at which the GPU out-runs 16 cores, for each batch size.

In [3]:
def break_even(batch, in_flight=1):
    grid = np.logspace(1, 7, 600); s = likelihood_speedup(grid, batch, in_flight=in_flight)
    hit = np.nonzero(s >= 1)[0]; return grid[hit[0]] if len(hit) else np.nan
B = np.array([1, 2, 4, 8, 16, 32, 64, 108, 256, 512, 1024, 4096])
fig = go.Figure(go.Scatter(x=B, y=[break_even(b) for b in B], mode="lines+markers",
                           line=dict(color=BLUE, width=2), marker=dict(size=9), name="break-even n"))
fig.add_vline(x=TRIALS_PER_ACCEPT, line_dash="dash", line_color=GREY,
              annotation_text="slice trials per accept", annotation_position="top")
fig.update_xaxes(type="log"); fig.update_yaxes(type="log")
style(fig, "Break-even data size vs batch size (A100 fp64, 16 CPU cores)", "evaluations per launch (B)",
      "data points needed for the GPU to win")

## 3. From likelihood speedup to wall clock — Amdahl

Even an infinitely fast likelihood leaves the rest of the sampler. The measured share `f`
is 0.53 at K=10 000 and 0.91 at K=64 000, so the ceiling is 2.1× and 11× respectively.
Larger K moves the ceiling up; so does a larger `n` (the likelihood share grows with it).

In [4]:
f = np.linspace(0.3, 0.99, 200)
fig = go.Figure()
for s, col, dash in [(2, GREY, "dot"), (4, GREY, "dash"), (8, BLUE, "solid"), (16, ORANGE, "solid")]:
    fig.add_trace(go.Scatter(x=f, y=end_to_end(s, f), name=f"likelihood {s}× faster", line=dict(color=col, width=2, dash=dash)))
for k, v in LIKELIHOOD_FRACTION.items():
    fig.add_vline(x=v, line_dash="dash", line_color=GREY, annotation_text=k, annotation_position="top")
style(fig, "End-to-end speedup vs the likelihood's share of run time", "likelihood share of CPU time (f)", "wall-clock speedup")

## 4. The levers, in the order they would be pulled

Cumulative: each bar adds the previous. Shown at `n = 1000` (the benchmark) and `n = 100 000`
(a realistic spectrum or image), with the K=64 000 likelihood share.

In [5]:
levers = [("today: 1 per launch",            dict(batch=1, in_flight=1)),
          ("+ batch the 16 candidates",         dict(batch=16)),
          ("+ batch slice trials (108)",         dict(batch=TRIALS_PER_ACCEPT)),
          ("+ 1024 candidates in flight",        dict(batch=1024))]
fig = go.Figure()
for n0, col in [(1000, BLUE), (100000, ORANGE)]:
    y = [end_to_end(likelihood_speedup(n0, **kw), LIKELIHOOD_FRACTION["K=64000"]) for _, kw in levers]
    fig.add_trace(go.Bar(x=[l for l, _ in levers], y=y, name=f"n = {n0:,}", marker_color=col,
                         text=[f"{v:.1f}×" for v in y], textposition="outside"))
fig.add_hline(y=1, line_dash="dash", line_color=GREY)
style(fig, "Expected wall-clock speedup by lever (A100 fp64 vs 16 cores, K=64000)", "", "wall-clock speedup")
fig.update_layout(barmode="group", bargap=0.3)

## 5. Which card

Saturated double-precision throughput measured on three generations with the same code.
The L40S's fp64 runs at 1/64 of its fp32 in hardware, yet costs only ~3× here: the kernel is
occupancy-bound, not fp64-bound. Double precision is affordable on every one of them.

In [6]:
cards = list(GPU_RATE_SAT); rates = [GPU_RATE_SAT[c] for c in cards]
fig = go.Figure(go.Bar(x=cards, y=rates, marker_color=[BLUE, GREY, GREY], text=[f"{r/1e6:.2f} M/s" for r in rates], textposition="outside"))
fig.add_hline(y=CPU_RATE_16C, line_dash="dash", line_color=ORANGE, annotation_text="CPU, 16 EPYC cores", annotation_position="top left")
style(fig, "Saturated likelihood throughput, fp64, 1000 points", "", "evaluations / s", h=380)

## What this says

* **Today's one-evaluation-per-launch design is roughly break-even on the 1000-point benchmark.**
  That is expected and is why Gate 0 (same evidence) matters more than its timing.
* **Batching the slice trials is the lever**: ~108 evaluations per launch amortises the launch
  cost, and the GPU line approaches its saturated 8× — on *this* data size.
* **Data size does the rest.** At 100 000 points the GPU wins even one evaluation at a time,
  and the likelihood's share of the run grows, lifting the Amdahl ceiling with it.
* The 100× lives in the combination — batching, more candidates in flight, and larger data —
  not in any single change. The model is here to say which one to pull next, with numbers.